# Gemma 4 E2B text-to-SQL benchmark (BIRD mini-dev + Spider)

Runs the team harness (`bench/`, from branch `kamalesh/smollm2-benchmark`, which contains the Qwen
ladder plus the small-model rungs) on a Colab T4 with **`gemma4:e2b-it-q4_K_M`** served by Ollama.
This is the same checkpoint family as `01_gemma_prompting.ipynb` (`google/gemma-4-E2B-it`), in the same
Q4_K_M quantization the Qwen and Phi branches use.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.

Protocol (same as the team): the repo's own splits (`data/processed/splits.json`), execution accuracy
with the repo scorer, one repair turn, temperature 0. Every setup is chosen on the **validation** sets
(`bird_val` 98, `spider_val` 100). The test sets (`bird_test` 155, `spider_test` 200) are run only for
the baseline and the setups picked on validation.

The long part (section 4) runs as a background process on the Colab VM and resumes from its own
predictions, so a browser disconnect does not stop it. Rerun the progress cell to check on it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
MODEL = 'gemma4:e2b-it-q4_K_M'
NUM_CTX = 16384        # same as the Qwen runs; not part of the LLM cache key, so keep it fixed
BRANCH = 'kamalesh/smollm2-benchmark'
COMMIT = '787ecda'     # pinned harness commit
WORK = '/content/gemma_bench'   # predictions, LLM cache, logs
USE_DRIVE = False      # True: keep WORK in Google Drive (asks for Drive access), survives a VM reset
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/gemma_bench'
os.makedirs(WORK, exist_ok=True)

## 1. Code, Python packages, Ollama

In [ ]:
%cd /content
!test -d repo/.git || git clone -q -b {BRANCH} https://github.com/sumonesphantom/CSE-572_Project_Skill_Evo.git repo
%cd /content/repo
!git checkout -q {COMMIT} && git log --oneline -1
!pip install -q -r requirements.txt

In [ ]:
!apt-get -qq install -y zstd pciutils lshw > /dev/null
!which ollama > /dev/null || (curl -fsSL https://ollama.com/install.sh | sh > /dev/null)
import subprocess, time, requests
OLLAMA_ENV = {**os.environ, 'OLLAMA_NUM_PARALLEL': '2', 'OLLAMA_CONTEXT_LENGTH': str(NUM_CTX), 'OLLAMA_KEEP_ALIVE': '-1'}

def ensure_ollama():
    # start `ollama serve` if it is not answering
    try:
        requests.get('http://localhost:11434/api/version', timeout=5); return
    except Exception:
        pass
    subprocess.Popen(['ollama', 'serve'], env=OLLAMA_ENV, stdout=open('/content/ollama.log', 'a'),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        try:
            print('ollama', requests.get('http://localhost:11434/api/version', timeout=5).json()); return
        except Exception:
            time.sleep(1)
    raise RuntimeError('ollama did not start; see /content/ollama.log')

ensure_ollama()
!ollama pull {MODEL} 2>&1 | tail -1
!ollama show {MODEL} | head -12

## 2. Data

In [ ]:
%cd /content/repo
import pathlib
RAW = pathlib.Path('/content/repo/data/raw')
if not (RAW / 'bird/minidev/MINIDEV/mini_dev_sqlite.json').exists():
    !mkdir -p {RAW}/bird {RAW}/spider
    !curl -sSL -o /content/minidev.zip https://bird-bench.oss-cn-beijing.aliyuncs.com/minidev.zip && unzip -q -o /content/minidev.zip -d {RAW}/bird && rm /content/minidev.zip
    !gdown -q 1403EGqzIDoHMdQF4c9Bkyl7dZLZ5Wt6J -O /content/spider.zip && unzip -q -o /content/spider.zip -d {RAW}/spider && rm /content/spider.zip
# card_games ships in WAL mode, which cannot be opened read-only (52 questions silently dropped otherwise)
import sqlite3, glob
for p in glob.glob('data/raw/bird/minidev/MINIDEV/dev_databases/*/*.sqlite'):
    c = sqlite3.connect(p)
    if c.execute('PRAGMA journal_mode').fetchone()[0].lower() == 'wal':
        print('fixed WAL:', p, c.execute('PRAGMA journal_mode=delete').fetchone())
    c.close()
# Colab's CPU is slow: one BIRD gold query needs more than the repo's 120 s limit, so prepare_data would drop
# that question (changing the splits) and the scorer would count it wrong. Give gold queries more time;
# the scoring rule itself is unchanged.
!sed -i 's/timeout_s=120)/timeout_s=1800)/' skillevo/sqlexec.py bench/metrics.py
!grep -n 'timeout_s=1800' skillevo/sqlexec.py bench/metrics.py
import subprocess, time
t0 = time.time()
!python scripts/prepare_data.py | grep -E "tasks|dropped"
print(f'prepare_data took {time.time() - t0:.0f}s')
dirty = subprocess.run(['git', 'status', '--short', 'data/'], capture_output=True, text=True).stdout.strip()
assert not dirty, 'data/processed differs from the committed splits: ' + dirty
print('splits match the committed ones')

In [ ]:
# predictions and the LLM cache live under WORK
%cd /content/repo
for local, remote in (('runs/bench', f'{WORK}/runs_bench'), ('data/cache', f'{WORK}/cache')):
    os.makedirs(remote, exist_ok=True)
    os.makedirs(os.path.dirname(local), exist_ok=True)
    !rm -rf {local} && ln -s {remote} {local}
!python -m pytest -q tests 2>&1 | tail -1

## 3. Smoke test (3 questions per set)

In [ ]:
ensure_ollama()
!python -m bench.run --model {MODEL} --strategy sm_pruned --sets bird_val,spider_val --num-ctx {NUM_CTX} --limit 3 --no-ves --workers 2

## 4. Validation ladder → selection → test sets → report (background)

Rungs (each changes one thing; see `bench/strategies.py`):

| Rung | Strategy | Adds |
|---|---|---|
| P1 | zs_repo | repo prompt (team baseline) |
| S1 | sm_extract | + robust SQL extraction (same prompt, reuses P1's answers) |
| S2 | sm_compact | + compact full schema, no 6,000-character cut |
| S3 | sm_pruned | + schema linking: top-4 tables + BIRD column descriptions |
| S6 | sm_pruned_fs3 | + 3 retrieved solved examples |
| P2 | zs_mschema | M-Schema (the Qwen branch's schema format) |
| P4m | ms_rules | + output rules |
| P5m | ms_fs3 | + 3 retrieved solved examples (Qwen's best setup) |
| S7 | sm_pruned_sc5 | S3 + best-of-5 vote on execution results (5× the cost, runs last) |

The pipeline picks the best rung on `bird_val`, on `spider_val` and on their mean, then runs the
baseline and those picks once on the test sets and writes the team CSV and tables.

In [ ]:
%%writefile /content/gemma_pipeline.py
"""Validation ladder -> selection -> test sets -> report. Resumable: rerun to continue."""
import json, os, subprocess, sys, time
import requests

MODEL, NUM_CTX, WORK = sys.argv[1], sys.argv[2], sys.argv[3]
REPO = '/content/repo'
RUNGS = ['zs_repo', 'sm_extract', 'sm_compact', 'sm_pruned', 'sm_pruned_fs3',
         'zs_mschema', 'ms_rules', 'ms_fs3', 'sm_pruned_sc5']
VAL = ['bird_val', 'spider_val']
TEST = ['bird_val', 'spider_val', 'bird_test', 'spider_test']
ENV = {**os.environ, 'OLLAMA_NUM_PARALLEL': '2', 'OLLAMA_CONTEXT_LENGTH': NUM_CTX, 'OLLAMA_KEEP_ALIVE': '-1'}
sys.path.insert(0, REPO)
from bench.run import run_name  # noqa: E402


def log(*a):
    print(time.strftime('%H:%M:%S'), *a, flush=True)


def ensure_ollama():
    try:
        requests.get('http://localhost:11434/api/version', timeout=5); return
    except Exception:
        pass
    subprocess.Popen(['ollama', 'serve'], env=ENV, stdout=open('/content/ollama.log', 'a'),
                     stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(60):
        try:
            requests.get('http://localhost:11434/api/version', timeout=5); return
        except Exception:
            time.sleep(1)
    raise RuntimeError('ollama did not start')


def metrics(rung):
    f = f'{REPO}/runs/bench/{run_name(MODEL, rung)}/metrics.json'
    return json.load(open(f))['metrics'] if os.path.exists(f) else {}


def complete(rung, sets):
    m = metrics(rung)
    return all(m.get(s) and not m[s].get('errors', {}).get('llm_error') for s in sets)


def bench(rung, sets):
    for attempt in range(3):   # a rerun retries questions that hit an Ollama server error
        if complete(rung, sets):
            return
        ensure_ollama()
        log(f'RUN {rung} on {",".join(sets)} (attempt {attempt + 1})')
        subprocess.run([sys.executable, '-m', 'bench.run', '--model', MODEL, '--strategy', rung,
                        '--sets', ','.join(sets), '--num-ctx', NUM_CTX, '--workers', '2'], cwd=REPO)
    log(f'WARNING {rung} still incomplete on {sets}')


log('PHASE validation')
for r in RUNGS:
    bench(r, VAL)
    m = metrics(r)
    if m.get('bird_val') and m.get('spider_val'):
        log(f"VAL {r:<16} bird_val {100 * m['bird_val']['ex']:.1f}  spider_val {100 * m['spider_val']['ex']:.1f}  "
            f"s/q {m['bird_val'].get('avg_llm_seconds') or 0:.1f}")

val = {r: metrics(r) for r in RUNGS}
val = {r: m for r, m in val.items() if m.get('bird_val') and m.get('spider_val')}
bird = lambda r: val[r]['bird_val']['ex']
spider = lambda r: val[r]['spider_val']['ex']
best_bird, best_spider = max(val, key=bird), max(val, key=spider)   # ties -> the earlier (simpler) rung
best_mean = max(val, key=lambda r: (bird(r) + spider(r)) / 2)
final = list(dict.fromkeys(['zs_repo', best_bird, best_spider, best_mean]))
json.dump({'model': MODEL, 'picked': {'bird': best_bird, 'spider': best_spider, 'overall': best_mean},
           'final_runs': final, 'validation': {r: {'bird_val': bird(r), 'spider_val': spider(r)} for r in val}},
          open(f'{WORK}/selection.json', 'w'), indent=2)
log('PICKED', 'bird:', best_bird, '| spider:', best_spider, '| overall:', best_mean)

log('PHASE test', final)
for r in final:
    bench(r, TEST)

log('PHASE report')
out = f'{REPO}/results/gemma4_e2b'
subprocess.run([sys.executable, '-m', 'bench.smol_report', '--prefix', MODEL.split(':')[0], '--out', out], cwd=REPO)
subprocess.run(['cp', f'{WORK}/selection.json', out])
subprocess.run(f'cd {REPO}/results && zip -qr {WORK}/gemma4_e2b_results.zip gemma4_e2b', shell=True)
log('DONE')

In [ ]:
# start the pipeline detached from the notebook (safe to rerun: it does nothing if one is already running)
import subprocess
assert not subprocess.run(['git', 'status', '--short', 'data/'], capture_output=True, text=True, cwd='/content/repo').stdout.strip(), 'splits differ from the committed ones: rerun the data cell'
running = subprocess.run(['pgrep', '-f', 'gemma_pipeline.py'], capture_output=True, text=True).stdout.split()
if running:
    print('pipeline already running, pid', running)
else:
    subprocess.Popen(['python', '-u', '/content/gemma_pipeline.py', MODEL, str(NUM_CTX), WORK], cwd='/content/repo',
                     stdout=open(f'{WORK}/pipeline.log', 'a'), stderr=subprocess.STDOUT, start_new_session=True)
    print('started; log:', f'{WORK}/pipeline.log')

## 5. Progress (rerun this cell any time)

In [ ]:
import subprocess
print('running:', bool(subprocess.run(['pgrep', '-f', 'gemma_pipeline.py'], capture_output=True, text=True).stdout.strip()))
!grep -E "PHASE|VAL |PICKED|WARNING|DONE|RUN " {WORK}/pipeline.log | tail -30
print('--- last lines')
!tail -4 {WORK}/pipeline.log
print('--- Ollama problems (empty = none)')
!grep -iE "panic|out of memory|fatal" /content/ollama.log | tail -5

## 6. Results (after the log shows DONE)

In [ ]:
%cd /content/repo
!cat {WORK}/selection.json
!cat results/gemma4_e2b/tables.md

In [ ]:
!cat results/gemma4_e2b/bench_results.csv

In [ ]:
# optional: download the team CSV, tables and per-question outputs
DOWNLOAD = False  # @param {type:"boolean"}
if DOWNLOAD:
    from google.colab import files
    files.download(f'{WORK}/gemma4_e2b_results.zip')